# 03 Geometric Perception And ICP

物体位姿不再给定时，机器人要从点云中估计刚体变换。

![ICP point cloud alignment visual map](assets/figures/03_geometric_perception_icp.png)

## Learning Objectives

- Describe ICP as alternating correspondence search and rigid transform fitting.
- Identify when geometric registration is a good fit for manipulation.
- Name the common failure modes caused by symmetry, occlusion, and poor initialization.

## Checkpoint

- Run the ICP example and identify the estimated rotation and translation.
- Explain why low mean error does not always mean the correct object pose.
- List one setup change that would make ICP more robust.

## Practice Task

Add noise or an outlier to the target points, rerun ICP, and write down how the error and estimated transform change.

## Concept Map

![Colab concept image](assets/colab/03_geometric_perception_icp_concept.png)

**Concept image.** ICP alternates between nearest-neighbor correspondences and a rigid transform fit.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


In [ ]:
import numpy as np
from rml.icp import icp
from rml.transforms import apply_transform, make_transform

source = np.array([[0.0, 0.0], [0.4, 0.1], [0.2, 0.7], [0.9, 0.6], [1.0, 0.0]])
target = apply_transform(make_transform(0.15, [0.2, -0.1]), source)
result = icp(source, target, iterations=30, tolerance=1e-10)
np.round(result.rotation, 3), np.round(result.translation, 3), result.mean_error

## Result Figure

Compare the original source points, target points, transformed source points, and nearest correspondences.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
distances = np.linalg.norm(result.transformed_source[:, None, :] - target[None, :, :], axis=2)
matches = target[np.argmin(distances, axis=1)]
fig, ax = plt.subplots()
ax.scatter(source[:, 0], source[:, 1], s=80, label='source', alpha=0.55)
ax.scatter(target[:, 0], target[:, 1], s=80, label='target')
ax.scatter(result.transformed_source[:, 0], result.transformed_source[:, 1], marker='x', s=100, label='transformed source')
for transformed_point, matched_point in zip(result.transformed_source, matches):
    ax.plot([transformed_point[0], matched_point[0]], [transformed_point[1], matched_point[1]], color='0.55', lw=1)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x position')
ax.set_ylabel('y position')
ax.legend(frameon=False)
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Change `noise_scale` and observe how registration error changes under deterministic target perturbations.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
rng = np.random.default_rng(7)
for noise_scale in [0.0, 0.02, 0.08]:
    noisy_target = target + rng.normal(0.0, noise_scale, size=target.shape)
    trial = icp(source, noisy_target, iterations=30, tolerance=1e-10)
    print('noise_scale=', noise_scale, 'mean_error=', round(trial.mean_error, 4))

## Reflection Prompt

什么时候 mean error 很低但姿态仍然可能不可信？结合对称物体、遮挡和初始化解释。

ICP 的失败模式：初始对齐太差、局部最优、点云遮挡、离群点、物体对称。

Exercise: 给 target 加一个离群点，观察 mean error 如何变化。